# ESUS validation: 60 prompts

Runs `extract_arch.py` on `prompts/validation/prompt_01.txt` through `prompt_60.txt` and records timing, token use, and readiness classes.

Inputs expected under `/kaggle/input`:

- `esus-code/`: this repository snapshot (including `prompts/validation`);
- `qwen3-4b-instruct/`: `Qwen3-4B-Instruct-2507-Q4_K_M.gguf`.

The metadata file uses placeholders for the Kaggle username and dataset names.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, urllib.request, zipfile

CODE_INPUT = Path('/kaggle/input/esus-code')
MODEL = Path('/kaggle/input/qwen3-4b-instruct/Qwen3-4B-Instruct-2507-Q4_K_M.gguf')
WORK = Path('/kaggle/working/esus')
RUN_ID = time.strftime('kaggle-%Y%m%d-%H%M%S')

assert CODE_INPUT.is_dir(), f'missing code dataset: {CODE_INPUT}'
assert MODEL.is_file(), f'missing model: {MODEL}'

if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(CODE_INPUT, WORK)
os.chdir(WORK)
print('repo:', WORK)
print('model:', MODEL)
print('run id:', RUN_ID)

In [ ]:
# Build llama.cpp only if the image does not already provide llama-server.
server_bin = shutil.which('llama-server')
build_dir = Path('/kaggle/working/llama.cpp')
if not server_bin:
    if not (build_dir / '.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/ggml-org/llama.cpp', str(build_dir)], check=True)
    subprocess.run(['cmake', '-S', str(build_dir), '-B', str(build_dir / 'build'),
                    '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF'], check=True)
    subprocess.run(['cmake', '--build', str(build_dir / 'build'),
                    '--config', 'Release', '-j2'], check=True)
    server_bin = str(build_dir / 'build' / 'bin' / 'llama-server')

assert Path(server_bin).is_file(), f'llama-server not found: {server_bin}'
print('llama-server:', server_bin)

In [ ]:
log = open('/kaggle/working/llama-server.log', 'w')
args = [server_bin, '-m', str(MODEL), '-c', '16384', '-fa', 'on',
        '--port', '8080', '-np', '1']
if shutil.which('nvidia-smi'):
    args[2:2] = ['-ngl', '99']
server = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
healthy = False
for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError(f'llama-server exited with {server.returncode}; see /kaggle/working/llama-server.log')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2) as response:
            if response.status == 200:
                healthy = True
                break
    except Exception:
        time.sleep(2)
if not healthy:
    server.terminate()
    raise RuntimeError('llama-server did not become healthy')
print('llama-server ready, pid', server.pid)

In [ ]:
run_dir = WORK / 'out' / 'validation' / 'runs' / RUN_ID
cmd = [sys.executable, 'scripts/run_validation.py',
       '--prompts', 'prompts/validation',
       '--out', str(run_dir.parent),
       '--run-id', RUN_ID,
       '--server-url', 'http://127.0.0.1:8080',
       '--timeout', '1800']
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK, check=True)
print('done:', run_dir)

In [ ]:
import csv
summary = run_dir / 'summary.csv'
with summary.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
print('prompts:', len(rows))
for row in rows:
    print(row['prompt'], row['classification'], row['seconds'],
          'out=' + row['tokens_out'], 'prompt=' + row['tokens_prompt'])

archive = shutil.make_archive(f'/kaggle/working/validation-{RUN_ID}', 'zip', run_dir)
print('archive:', archive)

## Notes

`scripts/run_validation.py` skips prompt directories with an existing `metrics.json` unless `--force` is used. To resume in a later Kaggle session, copy the previous `out/validation/runs/<run-id>` directory into the notebook or dataset and pass the same `--run-id`.

The classification and artifact definitions are documented in `doc/validation.md`.